# 🚀 TinyNav: Google Colab Training Notebook
Sử dụng notebook này để huấn luyện mô hình AI tự lái siêu tốc bằng GPU miễn phí của Google Colab!

**Các bước chuẩn bị trước khi chạy:**
1. Hãy đảm bảo bạn đã chuyển Runtime sang **T4 GPU** (Menu: *Runtime* -> *Change runtime type* -> Chọn *T4 GPU*).
2. Chạy cell tiếp theo để nhập thư viện.
3. Khi chạy cell upload, hãy bấm nút tải 2 file sau từ máy tính của bạn lên:
   - `preprocessed_data.npz` (trong thư mục `dataset/` trên laptop của bạn)
   - `model_baseline.keras` (trong thư mục `models/` trên laptop của bạn)
4. Nhấn *Runtime* -> *Run all* để tự động huấn luyện và tải file `best_model.keras` kết quả về máy.

In [ ]:
import os
import sys
import numpy as np
import tensorflow as tf
from google.colab import files
print("TensorFlow Version:", tf.__version__)
print("GPU co san:", tf.config.list_physical_devices('GPU'))

In [ ]:
print("Hãy bấm chọn tải lên 2 file 'preprocessed_data.npz' và 'model_baseline.keras'...")
uploaded = files.upload()

In [ ]:
# Định nghĩa các hàm Loss và Metric tùy chỉnh
def custom_loss(y_true, y_pred):
    # Tách nhãn steering, throttle, obstacle
    y_true_steer = y_true[:, 0:1]
    y_true_throt = y_true[:, 1:2]
    y_true_obst = y_true[:, 2:3]
    
    y_pred_steer = y_pred[:, 0:1]
    y_pred_throt = y_pred[:, 1:2]
    y_pred_obst = y_pred[:, 2:3]
    
    # Áp dụng activation thủ công để đồng bộ
    steer_pred_act = tf.tanh(y_pred_steer)
    throt_pred_act = tf.sigmoid(y_pred_throt)
    obst_pred_act = tf.sigmoid(y_pred_obst)
    
    steer_loss = tf.reduce_mean(tf.square(y_true_steer - steer_pred_act))
    throt_loss = tf.reduce_mean(tf.square(y_true_throt - throt_pred_act))
    obst_loss = tf.reduce_mean(tf.keras.losses.binary_crossentropy(y_true_obst, obst_pred_act))
    
    # w_steer = 1.0, w_throt = 1.0, w_obst = 0.5
    total_loss = steer_loss * 1.0 + throt_loss * 1.0 + obst_loss * 0.5
    return total_loss

def steer_mae(y_true, y_pred):
    return tf.reduce_mean(tf.abs(y_true[:, 0:1] - tf.tanh(y_pred[:, 0:1])))

def throt_mae(y_true, y_pred):
    return tf.reduce_mean(tf.abs(y_true[:, 1:2] - tf.sigmoid(y_pred[:, 1:2])))

def obst_accuracy(y_true, y_pred):
    pred_rounded = tf.round(tf.sigmoid(y_pred[:, 2:3]))
    return tf.reduce_mean(tf.cast(tf.equal(y_true[:, 2:3], pred_rounded), tf.float32))

In [ ]:
# Bộ sinh dữ liệu tăng cường (Data Generator)
def data_generator(X, Y_steer, Y_throt, Y_obst, batch_size=16, augment=True):
    num_samples = len(X)
    while True:
        indices = np.arange(num_samples)
        if augment:
            np.random.shuffle(indices)
            
        for start in range(0, num_samples, batch_size):
            end = min(start + batch_size, num_samples)
            batch_indices = indices[start:end]
            
            x_batch = X[batch_indices].astype(np.float32) / 255.0
            y_s_batch = Y_steer[batch_indices].copy()
            y_t_batch = Y_throt[batch_indices].copy()
            y_o_batch = Y_obst[batch_indices].copy()
            
            if augment:
                for i in range(len(x_batch)):
                    # 1. Flip
                    if np.random.rand() > 0.5:
                        x_batch[i] = np.flip(x_batch[i], axis=1)
                        y_s_batch[i] = -y_s_batch[i]
                        
                    # 2. Shift
                    dx = np.random.randint(-12, 13)
                    if dx > 0:
                        x_batch[i][:, dx:, :] = x_batch[i][:, :-dx, :]
                        x_batch[i][:, :dx, :] = x_batch[i][:, dx:dx+1, :]
                    elif dx < 0:
                        x_batch[i][:, :dx, :] = x_batch[i][:, -dx:, :]
                        x_batch[i][:, dx:, :] = x_batch[i][:, dx-1:dx, :]
                    y_s_batch[i] = np.clip(y_s_batch[i] + dx * 0.05, -1.0, 1.0)
                    
                    # 3. Brightness
                    brightness_factor = np.random.uniform(0.8, 1.2)
                    x_batch[i] = np.clip(x_batch[i] * brightness_factor, 0.0, 1.0)
                    
                    # 4. Contrast
                    contrast_factor = np.random.uniform(0.8, 1.2)
                    mean_val = np.mean(x_batch[i])
                    x_batch[i] = np.clip((x_batch[i] - mean_val) * contrast_factor + mean_val, 0.0, 1.0)
                    
                    # 5. Noise
                    if np.random.rand() > 0.5:
                        noise = np.random.normal(0, np.random.uniform(0.01, 0.04), x_batch[i].shape)
                        x_batch[i] = np.clip(x_batch[i] + noise, 0.0, 1.0)
            
            y_batch = np.column_stack([y_s_batch, y_t_batch, y_o_batch])
            yield x_batch, y_batch

In [ ]:
# Load và phân chia dữ liệu
print("Loading dataset...")
data = np.load("preprocessed_data.npz")
x_train, y_tr_steer, y_tr_throt, y_tr_obst = data["x_train"], data["y_train_steer"], data["y_train_throt"], data["y_train_obst"]
x_val, y_val_steer, y_val_throt, y_val_obst = data["x_val"], data["y_val_steer"], data["y_val_throt"], data["y_val_obst"]

print(f"Loaded {len(x_train)} train, {len(x_val)} val samples.")

In [ ]:
# Load mô hình và hỗ trợ Resume Training
BEST_MODEL = "best_model.keras"
BASE_MODEL = "model_baseline.keras"

if os.path.exists(BEST_MODEL):
    print(f"Tìm thấy mô hình đã học trước đó '{BEST_MODEL}'. Đang nạp để học tiếp...", flush=True)
    model = tf.keras.models.load_model(
        BEST_MODEL,
        custom_objects={
            "custom_loss": custom_loss,
            "steer_mae": steer_mae,
            "throt_mae": throt_mae,
            "obst_accuracy": obst_accuracy
        }
)
else:
    print(f"Nạp mô hình baseline mới từ '{BASE_MODEL}'...", flush=True)
    model = tf.keras.models.load_model(BASE_MODEL)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss=custom_loss,
    metrics=[steer_mae, throt_mae, obst_accuracy]
)
print("Model compiled successfully!")

In [ ]:
# Huấn luyện mô hình
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint

callbacks = [
    EarlyStopping(monitor='val_loss', patience=8, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=4, min_lr=1e-5, verbose=1),
    ModelCheckpoint(BEST_MODEL, monitor='val_loss', save_best_only=True, verbose=1)
]

BATCH_SIZE = 16
EPOCHS = 60

train_gen = data_generator(x_train, y_tr_steer, y_tr_throt, y_tr_obst, batch_size=BATCH_SIZE, augment=True)
val_gen = data_generator(x_val, y_val_steer, y_val_throt, y_val_obst, batch_size=BATCH_SIZE, augment=False)

history = model.fit(
    train_gen,
    steps_per_epoch=len(x_train) // BATCH_SIZE,
    validation_data=val_gen,
    validation_steps=len(x_val) // BATCH_SIZE,
    epochs=EPOCHS,
    callbacks=callbacks
)

In [ ]:
# Tải mô hình tốt nhất về máy tính
print("Đang tải file model tốt nhất 'best_model.keras' về máy tính của bạn...")
files.download(BEST_MODEL)